# Teste de conexão e endpoints - API Mercado Libre

Objetivo: validar o fluxo de autenticação (`client_credentials`) e descobrir **quais endpoints respondem hoje**, com qual status e com que formato, antes de ajustar o ETL.

## 1. Parâmetros e configuração

In [ ]:
import os
import time
from pathlib import Path
import pandas as pd
import requests
import yaml
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))          
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)


def find_config(name='config/config.yaml'):
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None


CONFIG_PATH = find_config()
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8')) if CONFIG_PATH else {}

BASE_URL = cfg.get('api', {}).get('base_url', 'https://api.mercadolibre.com').rstrip('/')
TIMEOUT = cfg.get('api', {}).get('timeout', 30)
SITE = cfg.get('search', {}).get('site', 'MLA')
QUERY = cfg.get('product', {}).get('query', 'S25 FE')
CATEGORY = cfg.get('product', {}).get('category', 'CELLPHONES')
if isinstance(QUERY, (list, tuple)):
    QUERY = QUERY[0]


DOMAIN_ID = f'{SITE}-{CATEGORY}'
TOKEN_PATH = cfg.get('api', {}).get('endpoints', {}).get('token', '/oauth/token')

print(f'config.yaml : {CONFIG_PATH or "não encontrado (usando padrões)"}')
print(f'API         : {BASE_URL}')
print(f'site / termo: {SITE} / {QUERY!r} / domínio {DOMAIN_ID}')

config.yaml : i:\mercado_livre\config\config.yaml
API         : https://api.mercadolibre.com
site / termo: MLA / 'S25 FE' / domínio MLA-CELLPHONES


## 2. Token de acesso

In [2]:
missing = [v for v in ('ML_CLIENT_ID', 'ML_CLIENT_SECRET') if not os.environ.get(v)]
if missing:
    raise EnvironmentError(f'Variáveis faltando no .env: {missing}')


def get_token():
    r = requests.post(
        f'{BASE_URL}{TOKEN_PATH}',
        data={
            'grant_type': 'client_credentials',
            'client_id': os.environ['ML_CLIENT_ID'],
            'client_secret': os.environ['ML_CLIENT_SECRET'],
        },
        timeout=TIMEOUT,
    )
    print('Status:', r.status_code)
    if r.status_code != 200:
        print(r.text[:500])
        return None

    data = r.json()
    print(f"Token obtido | tipo: {data.get('token_type')} | "
          f"expira em: {data.get('expires_in')}s | escopo: {data.get('scope')}")
    return data['access_token']


TOKEN = get_token()

Status: 200
Token obtido | tipo: Bearer | expira em: 21600s | escopo: offline_access read urn:ml:all:comunication:/read-write urn:ml:all:publish-sync:/read-write urn:ml:mktp:ads:/read-write urn:ml:mktp:comunication:/read-write urn:ml:mktp:invoices:/read-write urn:ml:mktp:metrics:/read-only urn:ml:mktp:offers:/read-write urn:ml:mktp:orders-shipments:/read-write urn:ml:mktp:publish-sync:/read-write urn:ml:vis:comunication:/read-write urn:ml:vis:publish-sync:/read-write write


## 3. Função auxiliar para testar endpoints

In [ ]:
results = []   

def call(label, path, params=None, auth=True):
    headers = {'Authorization': f'Bearer {TOKEN}'} if (auth and TOKEN) else {}
    started = time.perf_counter()
    status, body, detail = 'ERRO', None, ''

    try:
        r = requests.get(f'{BASE_URL}{path}', params=params, headers=headers, timeout=TIMEOUT)
        status = r.status_code
        try:
            body = r.json()
        except ValueError:
            body = None
        if status != 200:
            message = body.get('message') if isinstance(body, dict) else None
            detail = message or r.text[:200]
    except requests.RequestException as e:
        detail = str(e)

    ms = round((time.perf_counter() - started) * 1000)
    results.append({'teste': label, 'endpoint': path, 'status': status, 'ms': ms, 'detalhe': detail})

    flag = 'OK   ' if status == 200 else 'FALHA'
    print(f'[{flag}] {status} | {label} | {path} | {ms} ms' + (f' | {detail}' if detail else ''))
    return body


def multiget_summary(body):
    if not isinstance(body, list):
        return {}
    codes = [entry.get('code') or entry.get('status_code') for entry in body]
    return pd.Series(codes).value_counts().to_dict()


def attr(product, attribute_id):
    for a in product.get('attributes') or []:
        if a.get('id') == attribute_id:
            return a.get('value_name')
    return None

## 4. Endpoints do case

Os três endpoints indicados no enunciado para busca, moedas e conversão.

### 4.1 `/sites/{site}/search`

In [4]:
search = call('search do case', f'/sites/{SITE}/search',
              {'q': QUERY, 'condition': 'new', 'limit': 50, 'offset': 0})

[FALHA] 403 | search do case | /sites/MLA/search | 163 ms | forbidden


### 4.2 `/currencies`

In [5]:
currencies = call('currencies', '/currencies')
pd.DataFrame(currencies).head(10) if isinstance(currencies, list) else currencies

[OK   ] 200 | currencies | /currencies | 379 ms


,id,symbol,description,decimal_places
0,ARS,$,Peso argentino,2
1,BOB,Bs,Boliviano,2
2,BRL,R$,Real,2
3,CLF,UF,Unidad de Fomento,2
4,CLP,$,Peso Chileno,0
5,COP,$,Peso colombiano,0
6,CRC,₡,Colones,2
7,CUC,CUC,Peso Cubano Convertible,2
8,CUP,$,Peso Cubano,2
9,DOP,$,Peso Dominicano,2


### 4.3 `/currency_conversions/search`

In [6]:
conversion = call('currency_conversions', '/currency_conversions/search', {'from': 'ARS', 'to': 'USD'})
print('Campo "ratio" presente:', isinstance(conversion, dict) and 'ratio' in conversion)
conversion

[OK   ] 200 | currency_conversions | /currency_conversions/search | 156 ms
Campo "ratio" presente: True


{'currency_base': 'ARS',
 'currency_quote': 'USD',
 'ratio': 0.0006592,
 'rate': 0.0006592,
 'inv_rate': 1517,
 'creation_date': '2026-10-13T03:04:25.000+00:00',
 'valid_until': '2026-10-14T03:15:00.000+00:00'}

## 5. Fluxo de catálogo: `/products/search`

Alternativa quando o `/search` não está disponível: devolve **produtos de catálogo**, não publicações.

### 5.1 Busca simples (sem filtro de domínio)

In [7]:
products_basic = call('products/search (simples)', '/products/search',
                      {'status': 'active', 'site_id': SITE, 'q': QUERY})
print('paging:', (products_basic or {}).get('paging'))

[OK   ] 200 | products/search (simples) | /products/search | 2832 ms
paging: {'last': 'MLA80300157', 'limit': 10, 'offset': 0, 'total': 5795}


### 5.2 Busca com domínio e condição

In [8]:
search_params = {
    'q': QUERY,
    'domain_id': DOMAIN_ID,
    'site_id': SITE,
    'condition': 'new',
    'status': 'active',
    'limit': 50,
    'offset': 0,
}
products_search = call('products/search (domínio)', '/products/search', search_params)
print('paging:', (products_search or {}).get('paging'))

results_list = (products_search or {}).get('results', [])
pd.DataFrame([{
    'id': p.get('id'),
    'name': p.get('name'),
    'domain_id': p.get('domain_id'),
    'status': p.get('status'),
    'brand': attr(p, 'BRAND'),
    'line': attr(p, 'LINE'),
    'model': attr(p, 'MODEL'),
} for p in results_list]).head(10)

[OK   ] 200 | products/search (domínio) | /products/search | 568 ms
paging: {'last': 'MLA45502272', 'limit': 50, 'offset': 0, 'total': 235}


,id,name,domain_id,status,brand,line,model
0,MLA60827731,Samsung Galaxy S25 Fe 512gb Celeste,MLA-CELLPHONES,active,Samsung,Galaxy S,S25FE
1,MLA78875374,Samsung Galaxy S25 Fe Jetblack Negro,MLA-CELLPHONES,active,Samsung,Galaxy S25 FE,Galaxy S25 FE
2,MLA60250178,Samsung Galaxy S25 Fe 512gb Negro Intenso,MLA-CELLPHONES,active,Samsung,Galaxy S,Galaxy S25 FE
3,MLA64486693,Samsung Galaxy S25 Fe 256gb Celeste Celeste,MLA-CELLPHONES,active,Samsung,Galaxy S,S25FE
4,MLA64956086,Samsung S25 Fe 5g 8/256gb Color Negro,MLA-CELLPHONES,active,Samsung,S25 Fe,Samsung S25 Fe 5G
5,MLA75505937,Samsung Galaxy S25 Fe 256gb 8gb color negro,MLA-CELLPHONES,active,Samsung,Galaxy S,S25 FE
6,MLA64961932,Samsung S25 Fe 5g 8/256gb Color Blanco,MLA-CELLPHONES,active,Samsung,S25 Fe,Samsung S25 Fe 5G
7,MLA66224096,Celular Samsung Galaxy S25 Fe Jetblack 256gb,MLA-CELLPHONES,active,Samsung,Galaxy S,S25FE
8,MLA66706811,Samsung Galaxy S25 Fe 128gb Negro Intenso Negro,MLA-CELLPHONES,active,Samsung,Galaxy S,S25FE
9,MLA79321610,Samsung Galaxy S25 Fe 256gb Azul,MLA-CELLPHONES,active,Samsung,Galaxy S,Galaxy S25FE


### 5.3 Atributos do primeiro produto

In [9]:
if results_list:
    attributes = pd.DataFrame(results_list[0].get('attributes', []))
    display(attributes[['id', 'name', 'value_name']].head(15))

,id,name,value_name
0,BRAND,Marca,Samsung
1,LINE,Línea,Galaxy S
2,MODEL,Modelo,S25FE
3,IS_DUAL_SIM,Es Dual SIM,Sí
4,COLOR,Color,Celeste
5,INTERNAL_MEMORY,Memoria interna,512 GB
6,RAM,Memoria RAM,8 GB
7,SIM_CARD_SLOTS_NUMBER,Cantidad de ranuras para tarjeta SIM,1
8,ESIMS_NUMBER,Cantidad de eSIMs,1
9,OPERATING_SYSTEM_NAME,Nombre del sistema operativo,Android


### 5.4 Limite de paginação (`offset`)
Descobre até onde a API deixa paginar. Em testes anteriores o `offset` máximo aceito era 100.

In [10]:
for offset in (0, 50, 100, 150):
    call(f'products/search offset={offset}', '/products/search', {**search_params, 'offset': offset})

[OK   ] 200 | products/search offset=0 | /products/search | 190 ms
[OK   ] 200 | products/search offset=50 | /products/search | 544 ms
[OK   ] 200 | products/search offset=100 | /products/search | 471 ms
[FALHA] 400 | products/search offset=150 | /products/search | 176 ms | Invalid arguments for specific request. Please check details to satisfy validations


## 6. Publicações de um produto: `/products/{id}/items`

Escolhe automaticamente o primeiro produto que tenha publicações (muitos produtos de catálogo retornam 404).
Para fixar um produto, defina `PRODUCT_ID`.

In [ ]:
PRODUCT_ID = None    

candidates = [PRODUCT_ID] if PRODUCT_ID else [p['id'] for p in results_list[:10]]
product_items = None

for pid in candidates:
    body = call(f'products/{{id}}/items [{pid}]', f'/products/{pid}/items', {'limit': 50, 'offset': 0})
    if isinstance(body, dict) and body.get('results'):
        PRODUCT_ID, product_items = pid, body
        break

if product_items:
    listings = product_items['results']
    print(f'\nProduto escolhido: {PRODUCT_ID} | paging: {product_items.get("paging")}')
    display(pd.DataFrame(listings).head())
else:
    listings = []
    print('\nNenhum dos produtos testados retornou publicações.')

[OK   ] 200 | products/{id}/items [MLA60827731] | /products/MLA60827731/items | 208 ms

Produto escolhido: MLA60827731 | paging: {'total': 11, 'offset': 0, 'limit': 50}


,item_id,site_id,seller_id,accepts_mercadopago,price,category_id,currency_id,warranty,condition,listing_type_id,international_delivery_mode,tier,inventory_id,tags,deal_ids,official_store_id,original_price,shipping,seller_address,sale_terms,min_purchase_unit,user_product_id
0,MLA2556093906,MLA,2378399002,True,1055563.14,MLA1055,ARS,Sin garantía,new,gold_special,none,,,"[cbt_fulfillment_us, net_proceeds_prices, kvs_primary, standard_price_by_cha...",[],NaN,None,"{'free_shipping': True, 'store_pick_up': False, 'local_pick_up': False, 'mod...","{'city': {'id': '', 'name': 'China Grove'}, 'state': {'id': '', 'name': 'Tex...","[{'value_struct': None, 'id': 'WARRANTY_TYPE', 'name': 'Tipo de garantía', '...",1,NaN
1,MLA2117352013,MLA,165489568,True,1399884.00,MLA1055,ARS,Garantía del vendedor: 6 meses,new,gold_special,none,,,"[standard_price_by_quantity, kvs_primary, immediate_payment, cart_eligible]",[],NaN,None,"{'free_shipping': True, 'store_pick_up': False, 'local_pick_up': False, 'mod...","{'city': {'id': 'TUxBQ0NBUGNiZGQx', 'name': 'Córdoba'}, 'state': {'id': 'TUx...","[{'value_struct': None, 'id': 'WARRANTY_TYPE', 'name': 'Tipo de garantía', '...",1,MLAU5281554196
2,MLA2832009372,MLA,3087723457,True,1414000.00,MLA1055,ARS,Garantía del vendedor: 3 meses,new,gold_special,none,,,"[kvs_primary, immediate_payment, cart_eligible]",[],410550.0,None,"{'free_shipping': True, 'store_pick_up': False, 'local_pick_up': True, 'mode...","{'city': {'id': 'TUxBQ0NBUGZlZG1sYQ', 'name': 'PALERMO'}, 'state': {'id': 'T...","[{'value_struct': None, 'id': 'WARRANTY_TYPE', 'name': 'Tipo de garantía', '...",1,MLAU3768198837
3,MLA1774915413,MLA,3224029707,True,1415000.00,MLA1055,ARS,Garantía del vendedor: 3 meses,new,gold_special,none,,,"[kvs_primary, immediate_payment, cart_eligible]",[],NaN,None,"{'free_shipping': True, 'store_pick_up': False, 'local_pick_up': False, 'mod...","{'city': {'id': 'TUxBQ01PUmI1NTBj', 'name': 'Morón'}, 'state': {'id': 'TUxBU...","[{'value_struct': None, 'id': 'WARRANTY_TYPE', 'name': 'Tipo de garantía', '...",1,MLAU3934472253
4,MLA2593287362,MLA,378757741,True,1416000.00,MLA1055,ARS,Garantía del vendedor: 10 días,new,gold_special,none,,,"[kvs_primary, immediate_payment, cart_eligible]",[],NaN,None,"{'free_shipping': True, 'store_pick_up': False, 'local_pick_up': False, 'mod...","{'city': {'id': 'TUxBQ0NBUGNiZGQx', 'name': 'Córdoba'}, 'state': {'id': 'TUx...","[{'value_struct': {'number': 10, 'unit': 'días'}, 'id': 'WARRANTY_TIME', 'na...",1,MLAU3618619997


### 6.1 Campos de uma publicação (`shipping` e `sale_terms`)

In [12]:
if listings:
    first = listings[0]
    print('Campos disponíveis:', sorted(first.keys()))
    print('\nshipping :', first.get('shipping'))
    print('warranty :', first.get('warranty'))
    print('sale_terms:')
    display(pd.json_normalize(first.get('sale_terms') or []))

Campos disponíveis: ['accepts_mercadopago', 'category_id', 'condition', 'currency_id', 'deal_ids', 'international_delivery_mode', 'inventory_id', 'item_id', 'listing_type_id', 'min_purchase_unit', 'official_store_id', 'original_price', 'price', 'sale_terms', 'seller_address', 'seller_id', 'shipping', 'site_id', 'tags', 'tier', 'warranty']

shipping : {'free_shipping': True, 'store_pick_up': False, 'local_pick_up': False, 'mode': 'me2', 'logistic_type': 'fulfillment', 'tags': ['mandatory_free_shipping', 'cbt_fulfillment'], 'cost': 0}
warranty : Sin garantía
sale_terms:


,value_struct,id,name,value_name,value_id
0,None,WARRANTY_TYPE,Tipo de garantía,Sin garantía,6150835


## 7. Detalhe dos itens: `/items` e `/items/bulk`

Usa os `item_id` **reais** da etapa anterior (ids de produto de catálogo não funcionam aqui e retornam 404).
Serve também para verificar se o `/items` traz campos que o `/products/{id}/items` não traz, como `sold_quantity`.

In [13]:
item_ids = [l['item_id'] for l in listings[:20]]

if item_ids:
    items_multi = call('items (multiget)', '/items', {'ids': ','.join(item_ids)})
    print('códigos internos:', multiget_summary(items_multi))

    items_bulk = call('items/bulk', '/items/bulk', {'ids': ','.join(item_ids)})
    print('códigos internos:', multiget_summary(items_bulk))
else:
    items_multi = items_bulk = None
    print('Sem item_ids para testar.')

[OK   ] 200 | items (multiget) | /items | 197 ms
códigos internos: {403: 11}
[OK   ] 200 | items/bulk | /items/bulk | 174 ms
códigos internos: {403: 11}


In [ ]:
ok_items = [e['body'] for e in (items_multi or []) if e.get('code') == 200]

if ok_items and listings:
    extra = sorted(set(ok_items[0]) - set(listings[0]))
    print(f'Campos só no /items ({len(extra)}):', extra)
    print('\nsold_quantity disponível:', 'sold_quantity' in ok_items[0],
          '| valor:', ok_items[0].get('sold_quantity'))
else:
    print('Nenhum item retornou 200 em /items.')

Nenhum item retornou 200 em /items.


## 8. Vendedores: `/users`

In [15]:
seller_ids = sorted({l['seller_id'] for l in listings if l.get('seller_id')})[:20]

if seller_ids:
    sellers = call('users (multiget)', '/users', {'ids': ','.join(map(str, seller_ids))})
    print('códigos internos:', multiget_summary(sellers))

    columns = ['id', 'nickname', 'user_type', 'site_id', 'seller_reputation.level_id',
               'seller_reputation.transactions.total', 'status.site_status']
    bodies = [e['body'] for e in (sellers or []) if e.get('code') == 200]
    display(pd.json_normalize(bodies).reindex(columns=columns))
else:
    print('Sem seller_ids para testar.')

[OK   ] 200 | users (multiget) | /users | 198 ms
códigos internos: {200: 7}


,id,nickname,user_type,site_id,seller_reputation.level_id,seller_reputation.transactions.total,status.site_status
0,3224029707,FDATECNO,normal,MLA,5_green,1512,active
1,3087723457,GAMESENSEARGENTINA,brand,MLA,5_green,47481,active
2,2378399002,UNIVERSOELECTRONICOARF,normal,MLA,5_green,25558,active
3,582212326,BRUNELLAGHIONE,normal,MLA,4_light_green,6,active
4,165489568,FEDERICO BAGGINI,normal,MLA,5_green,742,active
5,246960746,BRENDA TOLABA,normal,MLA,5_green,79,active
6,378757741,GAJU1441810,normal,MLA,5_green,64,active


## 9. Resumo dos testes

- **200**: endpoint funciona.
- **401 / 403**: problema de token, permissão ou endpoint bloqueado. Compare com o fluxo de catálogo.
- **404**: recurso inexistente (em multiget, confira também os códigos internos).
- **400**: parâmetro inválido (veja a coluna `detalhe`).

In [16]:
summary = pd.DataFrame(results)
summary['resultado'] = summary['status'].map(lambda s: 'OK' if s == 200 else 'FALHA')
summary[['teste', 'endpoint', 'status', 'resultado', 'ms', 'detalhe']]

,teste,endpoint,status,resultado,ms,detalhe
0,search do case,/sites/MLA/search,403,FALHA,163,forbidden
1,currencies,/currencies,200,OK,379,
2,currency_conversions,/currency_conversions/search,200,OK,156,
3,products/search (simples),/products/search,200,OK,2832,
4,products/search (domínio),/products/search,200,OK,568,
5,products/search offset=0,/products/search,200,OK,190,
6,products/search offset=50,/products/search,200,OK,544,
7,products/search offset=100,/products/search,200,OK,471,
8,products/search offset=150,/products/search,400,FALHA,176,Invalid arguments for specific request. Please check details to satisfy vali...
9,products/{id}/items [MLA60827731],/products/MLA60827731/items,200,OK,208,
